In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# --- CONFIG ---
CLIENT_ID = os.environ.get("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.environ.get("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI = "http://127.0.0.1:8888/callback"
PLAYLIST_NAME = "Seedhe MauJ"
FILTER_ARTISTS = ["Seedhe Maut", "Calm", "Encore ABJ"]  # ← add/remove artists here

# --- AUTH ---
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope="user-library-read playlist-modify-public playlist-modify-private"
))

# --- FETCH LIKED SONGS & FILTER BY ARTISTS ---
filter_artists_lower = [a.lower() for a in FILTER_ARTISTS]
print(f"Fetching liked songs by: {FILTER_ARTISTS}")
filtered_uris = []
offset = 0
limit = 50

while True:
    results = sp.current_user_saved_tracks(limit=limit, offset=offset)
    tracks = results["items"]
    if not tracks:
        break
    for item in tracks:
        track = item["track"]
        track_artists = [a["name"].lower() for a in track["artists"]]
        # Add song if ANY of the filter artists appear in the track's artists
        if any(artist in track_artists for artist in filter_artists_lower):
            filtered_uris.append(track["uri"])
    offset += limit
    print(f"  Scanned {offset} songs, found {len(filtered_uris)} matches so far...")

print(f"Total matching songs: {len(filtered_uris)}")

if not filtered_uris:
    print("No songs found for these artists. Check the artist names and try again.")
    exit()

# --- CREATE NEW PLAYLIST ---
user_id = sp.current_user()["id"]
playlist = sp.current_user_playlist_create(PLAYLIST_NAME, public=True)
playlist_id = playlist["id"]
print(f"Created playlist: '{PLAYLIST_NAME}'")
playlist_id = playlist["id"]
print(f"Created playlist: '{PLAYLIST_NAME}'")

# --- ADD SONGS IN BATCHES OF 100 ---
for i in range(0, len(filtered_uris), 100):
    batch = filtered_uris[i:i+100]
    sp.playlist_add_items(playlist_id, batch)
    print(f"  Added songs {i+1} to {i+len(batch)}")

print("Done! Playlist created successfully.")

Fetching liked songs by: ['Seedhe Maut', 'Calm', 'Encore ABJ']
  Scanned 50 songs, found 20 matches so far...
  Scanned 100 songs, found 29 matches so far...
  Scanned 150 songs, found 63 matches so far...
  Scanned 200 songs, found 83 matches so far...
  Scanned 250 songs, found 100 matches so far...
  Scanned 300 songs, found 101 matches so far...
  Scanned 350 songs, found 102 matches so far...
  Scanned 400 songs, found 105 matches so far...
  Scanned 450 songs, found 105 matches so far...
  Scanned 500 songs, found 105 matches so far...
  Scanned 550 songs, found 105 matches so far...
  Scanned 600 songs, found 105 matches so far...
  Scanned 650 songs, found 105 matches so far...
  Scanned 700 songs, found 105 matches so far...
  Scanned 750 songs, found 106 matches so far...
  Scanned 800 songs, found 106 matches so far...
  Scanned 850 songs, found 106 matches so far...
  Scanned 900 songs, found 106 matches so far...
  Scanned 950 songs, found 106 matches so far...
  Scanned 1